# ⬢ Nosana Gridへの分散型OpenClawのデプロイ
### 分散型AIエージェントインフラの技術ガイド

このガイドでは、**OpenClaw**エージェントを**Nosana Grid**にデプロイする包括的な技術的ウォークスルーを提供します。従来のクラウドプロバイダーの制約（レイテンシ、検閲、高コスト）なしに、自律的なAIワークロードを実行するために分散型GPUノードを活用する方法を学びます。

--- 

**`[前提条件]`**
- **Nosana Dashboardへのアクセス:** [dashboard.nosana.com](https://dashboard.nosana.com)でサインアップしてください。
- **Solanaウォレット:** 取引手数料用に少量の**SOL**を保有する互換ウォレット（例：Phantom、Solflare）。
- **NOSトークン:** GridでのGPUコンピューティング時間の支払いに使用されます。
- **API認証情報:** （オプション）ハイブリッドモードを使用する場合、ClaudeやOpenAIなどの外部LLM用のAPIトークン。

**`[注]`**
Nosanaは市場主導のアプローチを採用しています。ハードウェア要件（GPU、VRAM）を定義すると、ネットワークが世界中のプロバイダーとマッチングします。

## ☰ 目次
1. [システムアーキテクチャ](#1.-System-Architecture)
2. [ジョブ定義の設定](#2.-Configuring-the-Job-Definition)
3. [モデル選択とVRAMの最適化](#3.-Model-Selection-&-VRAM-Optimization)
4. [デプロイメントプロセス](#4.-The-Deployment-Process)
5. [リアルタイム監視とログ分析](#5.-Real-time-Monitoring-&-Log-Analysis)
6. [ゲートウェイ接続と認証](#6.-Gateway-Connectivity-&-Authentication)
7. [外部API統合（ハイブリッドモード）](#7.-External-API-Integration)
8. [Telegram Bot統合](#8.-Telegram-Bot-Integration)
9. [プログラムによる対話（Python）](#9.-Programmatic-Interaction)
10. [トラブルシューティングとベストプラクティス](#10.-Troubleshooting-&-Best-Practices)

## 1. システムアーキテクチャ

Nosana上のOpenClawエコシステムは、モジュール化された分散パターンに従います：

1. **ユーザークライアント:** Telegram、Web UI、またはAPI呼び出しを介して対話します。
2. **OpenClawエージェントコンテナ:** タスクをオーケストレーションし、メモリを管理し、ツール呼び出しロジックを処理します。Nosana Grid上のコンテナ化されたワークロードとして実行されます。
3. **推論エンジン:** 通常、コンテナ内または隣接して実行される**vLLM**または**Ollama**で、LLM（例：GLM-4、Llama 3）を提供します。
4. **Nosana Grid:** デプロイメントをホストする分散型GPUプロバイダー（ノード）のネットワークです。

<img src="assets/architecture_diagram.png" width="700px" style="margin-top: 10px; border: 1px solid #ddd; padding: 5px;">
*図1: ユーザーと分散型GPUプロバイダー間の高レベルなインフラストラクチャフロー。*

## 2. ジョブ定義の設定

Nosanaでは、すべてのデプロイメントが**ジョブ定義**によって定義されます。OpenClawテンプレートを使用する場合、以下の技術パラメータが事前に設定されていますが、特殊なユースケースのために変更できます：

- **コンテナイメージ:** `openclaw/agent:latest`（またはカスタムフォーク）。
- **リソース:** 必要な最小CPUコア数、システムRAM、GPUスペックを定義します。
- **環境変数:** `API_KEYS`、`GATEWAY_TOKEN`、`BOT_TOKEN`などのシークレットに使用されます。
- **公開ポート:** ポート`8080`はOpenClaw APIゲートウェイの標準です。

## 3. モデル選択とVRAMの最適化

パフォーマンスとコスト効率のために適切なモデルを選択することは重要です。**VRAM（ビデオRAM）**が主なボトルネックとなります。

**`[GPU仕様マトリックス]`**
| モデル | パラメータ数 | 量子化 | 必要なVRAM | 推奨最小GPU |
| :--- | :--- | :--- | :--- | :--- |
| **GLM-4-9B-Chat** | 9B | FP16 / 4-bit | 18GB / 6GB | RTX 3090 (FP16) / RTX 3060 (4-bit) |
| **Llama-3-8B** | 8B | 4-bit (GGUF) | 8GB | RTX 3060 / 4060 |
| **DeepSeek-V3** | 671B | MoE (Mixed) | 160GB+ | 2x または 4x A100/H100 クラスター |

**`[プロのヒント]`** 大きなモデルをコンシューマー向けGPUで実行するには、**量子化モデル（4-bitまたは8-bit）**を使用して、大幅な性能低下なく動作させます。

<img src="assets/model_selection.png" width="600px">
*図2: Nosana Dashboardで適切なモデルとGPUノードを選択する。*

## 4. デプロイメントプロセス

デプロイメントを成功させるには、以下の手順に従ってください：

1. **Dashboardへのアクセス:** [dashboard.nosana.com/market](https://dashboard.nosana.com/market)にアクセスします。
2. **テンプレートの選択:** **OpenClaw**テンプレートを探して選択します。
3. **シークレットの設定:** **Secrets**セクションに`GATEWAY_TOKEN`（作成したカスタムパスワード）や外部APIキーを追加します。**プレーンテキストフィールドにキーを貼り付けないでください。**
4. **ノードのマッチング:** マーケットプレイスからGPUノードを選択します。**Price per Hour**と**Node Reputation**に注意してください。
5. **実行:** **Deploy**をクリックします。ネットワークは必要なNOSトークンをロックし、コンテナオーケストレーションを開始します。

<img src="assets/deployment_panel.png" width="600px">
*図3: デプロイメント用のシークレット設定とノード選択。*

## 5. リアルタイム監視とログ分析

デプロイメントがアクティブになったら、**Live Logs**を監視して、モデルがVRAMに正しくロードされることを確認してください。

**`[主要なログインジケーター]`**
- `[SYSTEM] Pulling Image...` -> コンテナがDocker Hubからダウンロード中です。
- `[CUDA] Device found: NVIDIA GeForce RTX 4090` -> GPUがコンテナに正常に渡されました。
- `[MODEL] Loading weights...` -> モデルがディスクからVRAMに移動中です。大きなモデルでは数分かかることがあります。
- `[READY] Server listening on 0.0.0.0:8080` -> エージェントがオンラインになり、リクエストを待機しています。

**警告:** `CUDA Out of Memory`が表示された場合、選択したモデルがノードのVRAMに対して大きすぎます。デプロイメントを停止し、より小さなモデルまたはVRAMが多いノードを選択してください。

## 6. ゲートウェイ接続と認証

Nosanaは、すべてのデプロイメントに安全な**Gateway URL**を提供します。アクセスは、手順4で定義したトークンによって制限されます。

1. **エンドポイントのコピー:** DashboardでURL（例：`https://node-xxx.nosana.ci`）を見つけます。
2. **認証:** ブラウザでURLを開くか、API経由で接続する際に、Authorizationヘッダーに`GATEWAY_TOKEN`を使用します。

<img src="assets/auth_screen.png" width="500px">
*図4: OpenClaw Gatewayでの認証。*

## 7. 外部API統合（ハイブリッドモード）

OpenClawは**Hybrid Intelligence**をサポートしています。Nosana上のローカルモデルを基本的なタスクに使用し、複雑な推論にはClaude 3.5 Sonnetなどの高-tierモデルに「フォールバック」できます。

**`[設定]`**
OpenClawの設定ファイルまたは環境変数に以下を追加します：
- `CLAUDE_API_KEY`: `sk-ant-xxx`
- `OPENAI_API_KEY`: `sk-proj-xxx`

これにより、エージェントはタスクの複雑さに基づいて動的にモデルを切り替えることができます。

## 8. Telegram Bot統合

AIエージェントをTelegramボットとして実行し、外出先でも簡単にアクセスできます。

1. **Botの作成:** [@BotFather](https://t.me/botfather)にメッセージを送信し、**HTTP API Token**を取得します。
2. **シークレットの注入:** `TELEGRAM_BOT_TOKEN`をNosanaデプロイメントのシークレットに追加します。
3. **権限の設定:** （オプション）`ALLOWED_TELEGRAM_USER_IDS`を追加して、自分だけがボットを使用できるようにします。
4. **開始:** ボットに`/start`を送信します。これで、Nosana Grid上のGPUを使用してメッセージを処理します。

<img src="assets/telegram_pairing.png" width="500px">
*図5: OpenClawエージェントとTelegramボットの正常なペアリング。*

## 9. プログラムによる対話（Python）

以下のスクリプトを使用して、分散型エージェントとプログラムで対話します。これは、エージェントを大規模な自動化パイプラインに統合するのに最適です。

In [ ]:
import requests
import json
import sys

# --- Configuration ---
# Replace with your actual Nosana Gateway URL and Token
GATEWAY_URL = "https://your-deployment-endpoint.nosana.ci"
GATEWAY_TOKEN = "your-custom-gateway-token"
MODEL_NAME = "glm-4-flash"  # Ensure this matches the model loaded on the node

def query_agent(prompt, stream=False):
    """Sends a prompt to the OpenClaw agent on Nosana Grid."""
    endpoint = f"{GATEWAY_URL}/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {GATEWAY_TOKEN}"
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "stream": stream
    }
    
    try:
        response = requests.post(endpoint, headers=headers, json=payload, stream=stream, timeout=60)
        response.raise_for_status()
        
        if stream:
            for line in response.iter_lines():
                if line:
                    # OpenClaw follows OpenAI stream format: 'data: {...}'
                    decoded_line = line.decode('utf-8')
                    if decoded_line.startswith('data: '):
                        data = json.loads(decoded_line[6:])
                        content = data['choices'][0]['delta'].get('content', '')
                        print(content, end='', flush=True)
        else:
            result = response.json()
            return result['choices'][0]['message']['content']
            
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to agent: {e}")
        return None

# Example usage:
if __name__ == "__main__":
    user_prompt = "Write a short summary of the benefits of decentralized AI compute."
    print(f"--- Querying Agent on Nosana Grid ---\nPrompt: {user_prompt}\n")
    
    # For non-streaming output:
    # response = query_agent(user_prompt)
    # print(f"Response: {response}")
    
    # For streaming output (recommended for long responses):
    query_agent(user_prompt, stream=True)

## 10. トラブルシューティングとベストプラクティス

**`[一般的な解決策]`**
- **モデルダウンロードの失敗:** GPUノードが安定したインターネット接続を持っていることを確認してください（通常はNosanaベンチマークで検証されます）。失敗した場合は、別のノードを試してください。
- **高レイテンシ:** 地理的に近いノード、または帯域評価の高いノードを選択してください。
- **ジョブの停止:** NOS残高が尽きると、ジョブは直ちに停止します。Dashboard設定で**Auto-Top-up**を有効にしてください。

**`[セキュリティのベストプラクティス]`**
- **シークレット管理:** すべてのキーにはNosana Secrets dashboardを使用してください。共有する場合は、ローカルスクリプトにハードコードしないでください。
- **レート制限:** 公開ボットをデプロイする場合は、NOSコストを管理するためにOpenClawの設定でレート制限を実装してください。
- **ノード選択:** 予期しないダウンタイムを避けるため、**Stability Score**の高いノードを優先してください。

--- 

**`[参考資料]`**
- [Nosana SDK Documentation](https://learn.nosana.com/)
- [OpenClaw Framework Repository](https://github.com/openclaw/openclaw)
- [Nosana Grid Status](https://status.nosana.com/)
- [Solana Explorer](https://explorer.solana.com/) （NOS/SOLトランザクションの追跡）